In [107]:
from datetime import date
import yfinance as yf
import pandas as pd
import numpy as np
import csv
from docplex.mp.model import Model
from docplex.mp.relaxer import Relaxer

In [108]:
def get_stocks_from_screener(start_date, end_date, filename = None):
  
  #load stocks from the screener files downloaded from Yahoo

  if filename == None:
    # I experimented with different assortments of stocks
    #df_russell = pd.read_csv("Russell_ex_lowvol.csv")           # 1000 higher volume stocks from the Russell 2000
    #df_sandp = pd.read_csv("SandP.csv")                         # S & P 500 (508)   
    #df_all_indexes = pd.read_csv("Eqities_all_indexes.csv")     # 626 stocks, includes all DOW, filtered S & P, filtered NASDAQ, filtered RUSSELL 2000
    
    df = pd.read_csv('The Dow The Dow The Dow Right Now.csv')    # DOW 30
    df_bonds = pd.read_csv("Bond ETFs.csv")                      # 60 Bond ETFs, includes government, corporate, a number of them don't have data
                                                                 # and cause yfinance errors, so they don't make it into the problem
    df_realestate = pd.read_csv("Real_Estate.csv")               # real estate stocks
    metals_etfs = ["SLV", "GLD"]                                 # 2 metal ETFs 
    quantum_stocks = ["QBTS", "IONQ", "RGTI", "QUBT" ]           # quantum computing stocks
   

    #df = pd.concat([df_all_indexes, df_bonds], axis = 0)
    df = pd.concat([df, df_bonds], axis = 0)
    df = pd.concat([df, df_realestate], axis = 0)  
    #Extract stock tickers from screener
    ticker_list = df['Symbol'].tolist()
    ticker_list.extend(metals_etfs)
    ticker_list.extend(quantum_stocks)
  else:
    df = pd.read_csv(filename)                                   # a new filename can be passed in
    ticker_list = df['Stock'].tolist()  
      
  ticker_string = " ".join(ticker_list)

  try:
    #Download historical data for all tickers at once
    data = yf.download(ticker_string, start = start_date, end = end_date, auto_adjust=True, group_by='ticker')
      
    #Check if the DataFrame is empty 
    if data.empty:
      print("Warning: No data was found for these dates or tickers.")
      data = None
    else:
      print("Successfully extracted prices.")
      data.to_csv('historical_data_results.csv')                 # write raw data to csv just in case, not used later in the code at this time
      print(data.head())                       

  except Exception as e:
     # Catches API errors
     #print(f"An unexpected error occurred: {e}")
     data = None

  return data  



In [109]:
def clean_data(data):
 
    # calculate daily returns as percentage and clean data, write the data to files that will be used by other functions  
      data = data.xs('Close', level='Price', axis=1)
      data = data.dropna(axis = 1)                             # drop rows that contain nan
      print("data")
      print(data.head())
      returns_df = data.pct_change().dropna()                  # drop rows that contain nan after pct_change
    
    
      threshold = 0.1
      cols_to_keep = returns_df.columns[(returns_df == 0).mean() <= threshold]   # drop columns with more than 10% 0s
      df_cleaned = data[cols_to_keep] 
      
      closing_prices = df_cleaned.iloc[-1]           #.xs('Close', level='Price')
      closing_prices.to_csv('lastprice_data.csv')                                # write closing price data to csv
      print("Last closing price data")
      print(closing_prices.head())

      returns_df_cleaned = returns_df[cols_to_keep]
      returns_df_cleaned.to_csv('returns_data.csv')                              # write returns data to csv
      print("Returns data")
      print(returns_df_cleaned.head())
 
      print(f"Number of stocks before filtering: {len(data.columns)}")
      print(f"Number of stocks after filtering: {len(df_cleaned.columns)}")

 

In [110]:

def get_tickers(verbose=False):
  #return a list of tickers from the lastprice_data file
  df = pd.read_csv('lastprice_data.csv')
  tickers = df['Ticker'].astype(str).tolist()
  if verbose:
      print(tickers)
      print(len(tickers))
  return tickers

def get_stock_info(coskew_compute = False, verbose=True):
  # Read in stock returns and price information from CSV
  df_price = pd.read_csv('lastprice_data.csv')
  
############### calculate mean returns #############################
  df_dailyreturn = pd.read_csv('returns_data.csv', index_col='Date')
  avg_daily_returns = df_dailyreturn.mean(axis=0)                  
  annual_returns = avg_daily_returns * 252
  returns = list(annual_returns)

############# calculate covariance ###################################
  covariance = (df_dailyreturn.cov() * 252).values.tolist()
    
############ calculate coskew tensor ###############################
  if coskew_compute == True:  
    returns_array = df_dailyreturn.to_numpy()
    print(returns_array)
      
    # Standardize the returns: (R - mu) / sigma 
    mu = np.mean(returns_array, axis=0)
    print("MEAN RETURNS ARRAY")
    print(mu)
    sigma = np.std(returns_array, axis=0)
    print("SIGMA ARRAY")  
    print(sigma)
    z_scores = (returns_array - mu) / sigma
    print("Z SCORES")
    print(z_scores)
      
    # Calculate the Coskewness Tensor 
    # Using Einstein Summation: 
    # t is day, i, j, k are individual stocks
    # Multiply z_i * z_j * z_k for every day and average.
    coskew_tensor = np.einsum('ti,tj,tk->ijk', z_scores, z_scores, z_scores) / returns_array.shape[0]

    print(f"Tensor Shape: {coskew_tensor.shape}")
    # Accessing S(X, Y, Z) for stocks 0, 1, and 2:
    print(f"Coskew (0,1,2): {coskew_tensor[0, 1, 2]:.4f}")  
  else:
    coskew_tensor = None
    
  if verbose: 
       print("Data Check")
       find_them = np.where(np.isnan(returns))[0]
       print(find_them)
       has_zero = 0 in covariance
       print(has_zero) 
       print("Daily return(the first 5 lines):")
       print(df_dailyreturn.head(5))
       print("Average daily return * 252:")
       print(returns)   
       print(f"Length of Daily returns: {len(returns)}")
       with open('output.csv', 'w', newline='') as f:
         writer = csv.writer(f)
         writer.writerow(covariance)
            
  return df_price, returns, covariance, coskew_tensor
    
def check_psd(matrix):
    # Check if matrix is positive definite
    # Calculate all eigenvalues
    eigenvalues = np.linalg.eigvals(matrix)
    
    # Get the smallest one
    min_eig = np.min(eigenvalues)
    
    # Check if it's non-negative (using a tiny tolerance for math noise)
    is_psd = min_eig >= -1e-10
    
    return is_psd, min_eig

In [111]:

def run_CPLEX():
  # Build and run the CPLEX model
    
  mdl = Model(name='portfolio_optimization')
  # Get stock names
  stock_names = get_tickers()
  
  # Get annualized stock data 
  # Price is stock closing price 2023-12-29
  price, returns, cov_matrix, coskew_tensor = get_stock_info(verbose=True)

  # Check that covariance is positive definitive
  # a, b = check_psd(cov_matrix)
  # print(a)
  # print(b)

  # n = number of stocks
  n = len(stock_names)
  
  # Create binary variables for each stock (0 = do not buy, 1 = buy)
  selection = mdl.binary_var_list(stock_names, name='select')

  # Create continuous variables for weights, upper bound is 5% - no single asset can make up more than 5% of the portfolio
  weights = mdl.continuous_var_list(stock_names, lb=0, ub=0.05, name='weight')

  # Link selection to weights
  # If selection[i] is 0, the weight must be 0.
  # If selection[i] is 1, the weight can be up to 1 (weight is fractional).
  for i in range(n):
    mdl.add_constraint(weights[i] <= selection[i])
       
  # Cardinality constraint
  # Pick an exact number of stocks out of the list
  mdl.add_constraint(mdl.sum(selection) == 30)

  # Weights must add to 1
  mdl.add_constraint(mdl.sum(weights) == 1, ctname='budget')

  # alternative code to set target return, use with risk minimizing objective function
  # mdl.add_constraint(mdl.sum(weights[i] * returns[i] for i in range(n)) >= target_return)

  # if a stock is selected, it should be allocated some minimum weight  
  #for i in range(n):
  #  mdl.add_constraint(weights[i] >= 0.01* selection[i])
    
  # cov_matrix = covariance matrix of asset returns
  variance = mdl.sum(weights[i] * weights[j] * cov_matrix[i][j] for i in range(n) for j in range(n))

  # risk factor must be high enough or the resulting portfolio will only have the few most profitable stocks
  risk_factor = 10
  return_factor = 1  

  # total return calculation 
  total_return = mdl.sum(weights[i] * returns[i] for i in range(n))

  # penalize if less than 100 stocks are selected  
  penalty_obj_num_stocks = 30 - sum( selection[i] for i in range(n))

  # penalize if weight is assigned to a stock that is not selected
  penalty_obj = sum( weights[i] *(1 - selection[i]) for i in range(n))

  #objective function to maximize returns, incorporating the total returns and covariance of stocks
  mdl.maximize(return_factor*total_return - risk_factor*variance - 1000*penalty_obj_num_stocks - 1*penalty_obj)

  # Alternative objective function to minimize risk, used when targeting a minimum return
  #mdl.minimize(variance)
  portfolio_data = []
  solution = mdl.solve(log_output=True) 
  if solution:
    print("Optimal Portfolio")
     
    for name, w_var, s_var in zip(stock_names, weights, selection):                # zip the names with the variables to iterate
       weight_percent = w_var.solution_value * 100
       stock = name[:6]
       portfolio_data.append({
           'Ticker': stock,
           'Weight': w_var.solution_value,
           'Weight_Pct': weight_percent,
           'Price' : price[price['Ticker'] == stock]['2023-12-29'].values[0],
           'Selection' : s_var.solution_value
       })
       
       if s_var.solution_value > 0.5:
         print(f"Stock: {stock} | Weight: {weight_percent:6.2f}%")                 # only print stocks that the solver selected (binary value = 1)
        
    df_portfolio = pd.DataFrame(portfolio_data)
    df_selections = df_portfolio[df_portfolio['Selection'] == 1 ]
    print(df_selections) 
    df_portfolio.to_csv('CPLEX_portfolio.csv')
    return df_portfolio, returns, cov_matrix, coskew_tensor
  else:
    print("No valid portfolio found.")
    return None
   


In [112]:
def backtest(df, start_date, end_date, budget):
  # Back testing
  # Build portolio with allocations and calculate return
  tickers = get_tickers()
  ticker_string = " ".join(tickers)
    
  # Download the start date prices and end date prices from yahoo
  start_price = yf.download(ticker_string, start = start_date, period = "5d", auto_adjust=True)
  end_price = yf.download(ticker_string, start = end_date, period = "5d", auto_adjust=True)
 
  # Extract the prices and merge into a dataframe
  df_start = start_price.xs('Close', level ='Price', axis = 1).iloc[[0]].stack().reset_index()
  df_start = df_start[['Ticker', 0]] 
  df_start.columns = ['Ticker', 'Close']
    
  df_end  = end_price.xs('Close', level ='Price', axis = 1).iloc[[0]].stack().reset_index()
  df_end = df_end[['Ticker', 0]]  
  df_end.columns = ['Ticker', 'Close']  

  df = df.merge(df_start, on='Ticker').rename(columns={'Close': 'Start_Price'})    # join on Ticker to make sure data goes with the right stock 
  df = df.merge(df_end, on='Ticker').rename(columns={'Close': 'End_Price'})

  # handles weights or discrete numbers of shares
  if df['Weight'].mean() < 1: 
    df['Shares'] = (df['Weight']* budget)/ df['Start_Price']  
  else:
    df['Shares'] = df['Weight']
      
  df['backtested_return'] = (df['End_Price'] - df['Start_Price'])* df['Shares']
  print(df)
  backtested_return = df['backtested_return'].sum()  
  print(f"Return over period from {start_date} to {end_date}: ${backtested_return:.2f}")
  return df, backtested_return





In [113]:
def risk(df, cov_matrix, budget, df_backtest):
# Calculate the risk of a portfolio
  if df['Weight'].mean() > 1: 
    df['Weight'] = (df['Weight']*df_backtest['End_Price'])/ budget  
      
  weights = df['Weight'].values
    
  # Calculate annualized Portfolio Variance (w^T * Cov * w)
  portfolio_variance = np.dot(weights.T, np.dot(cov_matrix, weights))

  # Use the standard deviation of the portfolio as the measure of risk
  portfolio_std_dev = np.sqrt(portfolio_variance)

  print(f"Annualized Portfolio Risk: {portfolio_std_dev:.2%}")
  return portfolio_std_dev


In [114]:
def generate_ampl_dat(tickers, returns, prices, cov_matrix, coskew_matrix, budget, target_skew, target_return,  filename="portfolio_BARON.dat"):
    # Generates a .dat file compatible with AMPL/NEOS for portfolio optimization.

    n = len(tickers)
    with open(filename, "w") as f:
 
        # Tickers
        f.write(f"set ASSETS := {' '.join(tickers)};\n\n")

        f.write(f"param budget := {budget};\n\n")
        
        # Target Parameters
        f.write(f"param target_skew := {target_skew};\n\n")
        #f.write(f"param target_return := {target_return};\n\n")

        # Price data
        f.write("param price:=\n")
        for ticker, price in zip(tickers, prices):
            f.write(f"    {ticker:<10} {price:8.4f}\n")
        f.write(";\n\n")

        # coSkew data
        f.write("param coskew :=\n")
        for i, ticker_i in zip(range(len(tickers)), tickers):
          for j, ticker_j in zip(range(len(tickers)), tickers):  
            for k, ticker_k in zip(range(len(tickers)), tickers): 
                val = coskew_matrix[i, j, k]
                f.write(f"{ticker_i} {ticker_j} {ticker_k} {val:.8f}\n")
        f.write(";\n")

        # Returns data
        f.write("param return_ind :=\n")
        for ticker, ret in zip(tickers, returns):
            f.write(f"    {ticker:<10} {ret:8.4f}\n")
        f.write(";\n\n")
        
        # Covariance Matrix (Formatted with headers)
        header_row = " ".join([f"{t:>12}" for t in tickers])
        f.write(f"param cov : {header_row} :=\n")
        for i, ticker_row in enumerate(tickers):
            # Extract row from matrix and format each value
            row_values = " ".join([f"{val:12.8f}" for val in cov_matrix[i]])
            f.write(f"    {ticker_row:<10} {row_values}\n")
        
        f.write(";\n")
    print(f"Successfully generated {filename}")


In [115]:
# Main cell, calls functions to load data, build and run the CPLEX model, and back test 

# Set start and end dates for historical optimization data
start_date = '2021-01-01'
end_date = '2024-01-01'

# Write stock data to files
data = get_stocks_from_screener(start_date, end_date)

#data = pd.read_csv("historical_data_results.csv", header=[0, 1], index_col=0)

#Remove NaNs, stocks with 0 returns 
clean_data(data)

# Run CPLEX portfolio optimization, data is loaded from files written by get_stocks_from_screener
df, returns, cov, coskew_tensor = run_CPLEX()
print(df)

# Set start and end dates for back test
start_date = '2024-01-01'

#today = date.today()
#end_date = today.strftime('%Y-%m-%d')
end_date = '2026-04-02'

# Run backtest
budget = 1000000          
df_backtest, portfolio_return = backtest(df, start_date, end_date, budget)
portfolio_return_percentage = portfolio_return/budget

# Calculate portfolio risk
risk_level = risk(df, cov, budget, df_backtest)
risk_free_rate = 0.04                      # 4.0% rate is typical safe investment benchmark

# Calculate sharpe ratio
sharpe_ratio = (portfolio_return_percentage - risk_free_rate) / risk_level

# Output results - risk is printed in risk() function
print(f"Portfolio return percentage: {portfolio_return_percentage:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
df_backtest.rename(columns={'backtested_return': 'Profit'})
df_selections = df_backtest[df_backtest['Selection'] == 1 ].copy()
df_selections['Cost'] = df_selections['Start_Price']*df_selections['Shares']
df_selections['Portion of Portfolio'] = df_selections['Price'] *df_selections['Shares']

#Write results to file
df_selections.to_csv('CPLEX_NEW_OUTPUT.csv')



[*                      2%                       ]  2 of 125 completed$ZTWO: possibly delisted; no price data found  (1d 2021-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1704085200")
$AOHY: possibly delisted; no price data found  (1d 2021-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1704085200")
[*                      3%                       ]  4 of 125 completed$NUSB: possibly delisted; no price data found  (1d 2021-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1704085200")
[*                      3%                       ]  4 of 125 completed$DUKH: possibly delisted; no price data found  (1d 2021-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200, endDate = 1704085200")
$MULT: possibly delisted; no price data found  (1d 2021-01-01 -> 2024-01-01) (Yahoo error = "Data doesn't exist for startDate = 1609477200

Successfully extracted prices.
Ticker     SDSI                             SKOR                        \
Price      Open High Low Close Volume       Open       High        Low   
Date                                                                     
2021-01-04  NaN  NaN NaN   NaN    NaN  44.958159  44.998366  44.933540   
2021-01-05  NaN  NaN NaN   NaN    NaN  44.926962  44.958145  44.925322   
2021-01-06  NaN  NaN NaN   NaN    NaN  44.843273  44.844093  44.843273   
2021-01-07  NaN  NaN NaN   NaN    NaN  44.820299  44.835067  44.802244   
2021-01-08  NaN  NaN NaN   NaN    NaN  44.777623  44.777623  44.736596   

Ticker                         ...  TMB                                  HYXF  \
Price           Close  Volume  ... High Low Close Adj Close Volume       Open   
Date                           ...                                              
2021-01-04  44.990982    3700  ...  NaN NaN   NaN       NaN    NaN  38.972915   
2021-01-05  44.958145    7700  ...  NaN NaN   NaN   

[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed


   Ticker    Weight  Weight_Pct       Price  Selection  Start_Price  \
0    SKOR  0.000000    0.000000   43.065105        0.0    42.894524   
1     MAA  0.000000    0.000000  120.806038        0.0   122.045883   
2    USTB  0.050000    5.000000   44.469894        1.0    44.335373   
3    USIG  0.000000    0.000000   46.261406        0.0    46.035778   
4     VTC  0.000000    0.000000   69.797058        1.0    69.418770   
..    ...       ...         ...         ...        ...          ...   
75    EXR  0.006848    0.684806  145.419983        1.0   149.111481   
76    TRV  0.046995    4.699482  183.416824        1.0   184.312286   
77   FALN  0.000000    0.000000   22.902250        1.0    22.798147   
78    MRK  0.050000    5.000000  101.526337        1.0   105.456276   
79   HYXF  0.000000    0.000000   39.459122        1.0    39.395775   

     End_Price       Shares  backtested_return  
0    48.570000     0.000000           0.000000  
1   123.367676     0.000000           0.000000  


In [96]:
# Try problem using BARON solver which can handle third order skew constraints

tickers = get_tickers()
df_price, returns, cov_matrix, coskew_tensor = get_stock_info(verbose=True, coskew_compute = True)
budget = 1000000

# Generate AMPL .dat file
price = df_price['2023-12-29'].tolist()
generate_ampl_dat(tickers, returns, price, cov_matrix, coskew_tensor, budget, target_skew = -0.15, target_return = 0.30)

#submit generated AMPL files to NEOS

[[-0.00032589 -0.00072935 -0.00266497 ...  0.00204433 -0.01099292
  -0.00038075]
 [ 0.01654815 -0.00255534 -0.00705425 ... -0.00306028 -0.03181291
  -0.001142  ]
 [-0.00120258 -0.00073183 -0.00032326 ...  0.00477498 -0.01108479
   0.00228682]
 ...
 [ 0.00186506  0.00493663  0.00750295 ...  0.00264652  0.0025615
   0.00471988]
 [ 0.01667758 -0.00131121 -0.00263201 ... -0.00263954  0.00068132
  -0.0018568 ]
 [-0.01530558 -0.00027089 -0.00238182 ... -0.00189042  0.00306386
  -0.00453178]]
MEAN RETURNS ARRAY
[ 3.47672970e-04 -5.33458026e-05 -1.18410962e-04 -1.11837381e-04
  1.12334126e-03 -1.24820018e-04  5.77704535e-05  4.95176087e-04
 -2.23675382e-04  3.07770934e-04  8.64271063e-05  6.11028923e-04
  1.20547183e-05  2.14022905e-04  4.71166631e-04  2.32005015e-03
 -8.36064128e-06  3.83713780e-04  7.11985682e-04  1.17954087e-04
  1.02265875e-04 -2.85957373e-04  5.31458330e-04  1.06627406e-03
  7.36923790e-04 -1.95725742e-05  9.00476067e-04  3.69293286e-04
 -3.13265521e-06 -1.36836086e-04 -3

In [118]:
#Process BARON solver output returned on NEOS
budget = 1000000

# Ticker_Weight.csv is created by copying and pasting the BARON output into a file
df_BARON = pd.read_csv("Ticker_Weight.csv")
df_backtest1, portfolio_return_BARON = backtest(df_BARON, start_date, end_date, budget)
portfolio_return_percentage_BARON = portfolio_return_BARON/budget

# Calculate portfolio risk
risk_level_BARON = risk(df_BARON, cov, budget, df_backtest1)
risk_free_rate = 0.04                      # 4.0% rate is typical safe investment benchmark
sharpe_ratio_BARON = (portfolio_return_percentage_BARON - risk_free_rate) / risk_level_BARON
print(f"Portfolio return percentage: {portfolio_return_percentage_BARON:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio_BARON:.2f}")
df_backtest1.to_csv("BARON_data.csv")

[*********************100%***********************]  80 of 80 completed
[*********************100%***********************]  80 of 80 completed


   Ticker  Select  Weight  Start_Price   End_Price  Shares  backtested_return
0     MAA       0       0   122.045860  123.367676       0           0.000000
1    SKOR       0       0    42.894531   48.570000       0           0.000000
2     VTC       0       0    69.418762   76.828003       0           0.000000
3    USIG       0       0    46.035782   51.209999       0           0.000000
4     SPG       1     150   128.010574  188.669998     150        9098.913574
..    ...     ...     ...          ...         ...     ...                ...
75    MRK       1    1200   105.456276  120.870003    1200       18496.472168
76    TRV       0       0   184.312286  293.989990       0           0.000000
77   FALN       1       1    22.798147   26.783001       1           3.984854
78     KO       0       0    55.998600   76.720001       0           0.000000
79   HYXF       0       0    39.395790   46.311001       0           0.000000

[80 rows x 7 columns]
Return over period from 2024-01-01 to 202

In [105]:
def backtest_cqm_nl(filename):

  # This function is for backtesting the results from the D-Wave CQM
  budget = 1000000
  df_dwave_cqm = pd.read_csv(filename)
  df_dwave_cqm = df_dwave_cqm.rename(columns = {'Stock':'Ticker', 'Shares':'Weight'})
  tickers = df_dwave_cqm['Ticker'].tolist()
  ticker_string = " ".join(tickers)
  df = df_dwave_cqm
  # Set start and end dates for back test
  start_date = '2024-01-01'
  
  #today = date.today()
  #end_date = today.strftime('%Y-%m-%d')
  end_date = '2026-04-02'    
  start_price = yf.download(ticker_string, start = start_date, period = "5d", auto_adjust=True)
  end_price = yf.download(ticker_string, start = end_date, period = "5d", auto_adjust=True)
   
  df_start = start_price.xs('Close', level ='Price', axis = 1).iloc[[0]].stack().reset_index()
  df_start = df_start[['Ticker', 0]] 
  df_start.columns = ['Ticker', 'Close']
      
  df_end  = end_price.xs('Close', level ='Price', axis = 1).iloc[[0]].stack().reset_index()
  df_end = df_end[['Ticker', 0]]  
  df_end.columns = ['Ticker', 'Close']  
  
  df = df.merge(df_start, on='Ticker').rename(columns={'Close': 'Start_Price'})    # join on Ticker to make sure data goes with the right stock 
  df = df.merge(df_end, on='Ticker').rename(columns={'Close': 'End_Price'})
      
  df['Shares'] = df['Weight']
  
  print(df.head())
  print(df.info())
  
  df['backtested_return'] = (df['End_Price'] - df['Start_Price'])* df['Shares']
  print(df)
  backtested_return = df['backtested_return'].sum()  
  print(f"Return over period from {start_date} to {end_date}: ${backtested_return:.2f}")
  
  #df_backtest1, portfolio_return_CQM = backtest(df_dwave_cqm, start_date, end_date, budget)
  portfolio_return_percentage_CQM = backtested_return/budget

  data = get_stocks_from_screener(start_date, end_date, filename = filename)
  clean_data(data)
  price, returns, cov_matrix, coskew_tensor = get_stock_info(verbose=True)

  # Calculate portfolio risk
  risk_level_CQM = risk(df_dwave_cqm, cov_matrix, budget, df)
  risk_free_rate = 0.04                      # 4.0% rate is typical safe investment benchmark
  sharpe_ratio_CQM = (portfolio_return_percentage_CQM - risk_free_rate) / risk_level_CQM
  print(f"Portfolio return percentage: {portfolio_return_percentage_CQM:.2%}")
  print(f"Sharpe Ratio: {sharpe_ratio_CQM:.2f}")
  print(price.head())
  df['2026-04-01'] = price['2026-04-01']
  df['Profit'] = df['End_Price'] * df['Weight'] - df['Start_Price'] * df['Weight']
  df.to_csv("CQM_data.csv")

In [106]:
# call the function to backtest results downloaded from the CQM
filename = "portfolio_data_NL.csv"
backtest_cqm_nl(filename)

[*********************100%***********************]  20 of 20 completed
[*********************100%***********************]  20 of 20 completed
[*****                 10%                       ]  2 of 20 completed

  Ticker  Weight  2023-12-29  Cost of Shares  Start_Price   End_Price  Shares
0    VTC      74   69.797066     5164.982864    69.418747   76.828003      74
1   USTB    6023   44.469887   267842.128075    44.335365   50.429317    6023
2   SLQD    3052   44.979969   137278.865463    44.888550   50.380001    3052
3   AMZN       1  151.940002      151.940002   149.929993  209.770004       1
4    AMH     109   33.368999     3637.220943    33.907211   29.080000     109
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Ticker          20 non-null     object 
 1   Weight          20 non-null     int64  
 2   2023-12-29      20 non-null     float64
 3   Cost of Shares  20 non-null     float64
 4   Start_Price     20 non-null     float64
 5   End_Price       20 non-null     float64
 6   Shares          20 non-null     int64  
dtypes: float64(4), int64

[*********************100%***********************]  20 of 20 completed


Successfully extracted prices.
Ticker           IGSB                                                 USTB  \
Price            Open       High        Low      Close   Volume       Open   
Date                                                                         
2024-01-02  46.444276  46.480604  46.435196  46.435196  1961400  44.407115   
2024-01-03  46.389782  46.444271  46.344371  46.417027  1982600  44.353306   
2024-01-04  46.380697  46.389780  46.344369  46.371616  3556300  44.353311   
2024-01-05  46.317120  46.462428  46.317120  46.344364  2903900  44.317437   
2024-01-08  46.371609  46.480590  46.371609  46.426098  2117300  44.326411   

Ticker                                              ...          HD  \
Price            High        Low      Close Volume  ...        Open   
Date                                                ...               
2024-01-02  44.407115  44.209828  44.335373  65700  ...  325.630852   
2024-01-03  44.362275  44.299503  44.353306  34000  ...  323